# Investigate criticality of anycast prefixes

## how?
* using Savvas' prefix-top-list (PAM'26 paper)

## TODOs
* get more details on these top lists
* do we need to look at DNS toplists (tranco/radar) ourselves?
* what about the top-as-list?

In [6]:
import pandas as pd
from pathlib import Path

import sys
analysis_dir = Path.cwd().parent
sys.path.append(str(analysis_dir))
from add_ASN import CaidaASLookup
import census_helper
from datetime import datetime

ts = datetime(2026, 2, 3)

/Users/remi/.venvs/default/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## load in criticality datasets

In [23]:
# top_as = pd.read_csv('as_top_list_ranked.csv')
# top_as

In [3]:
top_prefix = pd.read_csv('prefix_top_list_ranked.csv')
top_prefix

,prefix,weight,domains,ips,ases
0,2a00:1450:400e::/48,5.820223e-02,"0-ai--zenmele-pages-dev-0.cdn.ampproject.org, ...","2a00:1450:400e:10::7, 2a00:1450:400e:10::8, 2a...",15169
1,142.250.179.0/24,2.332626e-02,"0-ai--zenmele-pages-dev-0.cdn.ampproject.org, ...","142.250.179.110, 142.250.179.115, 142.250.179....",15169
2,57.144.222.0/23,1.927785e-02,"0.facebook.com, 0.freebasics.com, 5-edge-chat....","57.144.222.1, 57.144.222.128, 57.144.222.129, ...",32934
3,2a03:2880:f36f::/48,1.920212e-02,"0.facebook.com, 0.freebasics.com, 5-edge-chat....","2a03:2880:f36f:120:face:b00c:0:167, 2a03:2880:...",32934
4,2a02:26f0:1180::/48,1.789608e-02,"0.ecom.attccc.com, 0.samsung-internet.filter-l...","2a02:26f0:1180:180::1018, 2a02:26f0:1180:180::...",20940
...,...,...,...,...,...
134654,2402:9400::/32,6.061169e-11,site24x7.enduserexp.com,2402:9400:400:2::204,55803
134655,2403:7000:8000::/34,6.061169e-11,site24x7.enduserexp.com,2403:7000:8000:700::83,45179
134656,173.249.152.0/24,6.061169e-11,site24x7.enduserexp.com,173.249.152.22,36444
134657,179.49.5.0/24,6.061169e-11,site24x7.enduserexp.com,179.49.5.66,22724


## load in anycast prefixes

In [16]:
census = census_helper.download_date(ts.year, ts.month, ts.day, 'v4')
# filter on GCD-confirmed (high-confidence)
census = census[census['GCD_ICMPv4'] > 1]
# filter on relevant columns
census = census[['prefix', 'backing_prefix', 'GCD_ICMPv4', 'locations']]

census.head()

,prefix,backing_prefix,GCD_ICMPv4,locations
0,1.0.0.0/24,1.0.0.0/24,75,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
1,1.1.1.0/24,1.1.1.0/24,74,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
3,1.10.10.0/24,1.10.10.0/24,3,"[{'city': 'Mumbai', 'country_code': 'IN', 'id'..."
4,1.12.0.0/24,1.12.0.0/20,5,"[{'city': 'Baltimore', 'country_code': 'US', '..."
5,1.12.12.0/24,1.12.0.0/20,3,"[{'city': 'Hong Kong', 'country_code': 'HK', '..."


In [17]:
print(f"Number of GCD-confirmed /24s {len(census):,}")
print(f"number of backing anycast prefixes {census['backing_prefix'].nunique():,}")

Number of GCD-confirmed /24s 14,332
number of backing anycast prefixes 5,148


## Join datasets

In [18]:
census_critical = pd.merge(
    top_prefix,
    census,
    left_on='prefix',
    right_on='backing_prefix',
    how='inner',
)

census_critical.head()

,prefix_x,weight,domains,ips,ases,prefix_y,backing_prefix,GCD_ICMPv4,locations
0,151.101.204.0/22,0.009964,"0.testing.pinterest.com, 01.emailinboundproces...","151.101.204.102, 151.101.204.106, 151.101.204....",54113,151.101.204.0/24,151.101.204.0/22,2,"[{'city': 'New York', 'country_code': 'US', 'i..."
1,151.101.204.0/22,0.009964,"0.testing.pinterest.com, 01.emailinboundproces...","151.101.204.102, 151.101.204.106, 151.101.204....",54113,151.101.205.0/24,151.101.204.0/22,2,"[{'city': 'New York', 'country_code': 'US', 'i..."
2,151.101.204.0/22,0.009964,"0.testing.pinterest.com, 01.emailinboundproces...","151.101.204.102, 151.101.204.106, 151.101.204....",54113,151.101.206.0/24,151.101.204.0/22,2,"[{'city': 'New York', 'country_code': 'US', 'i..."
3,151.101.204.0/22,0.009964,"0.testing.pinterest.com, 01.emailinboundproces...","151.101.204.102, 151.101.204.106, 151.101.204....",54113,151.101.207.0/24,151.101.204.0/22,2,"[{'city': 'New York', 'country_code': 'US', 'i..."
4,162.159.128.0/19,0.008800,"01net.it, 02db8c58be312b9a93c947918f83eae8e006...","162.159.128.102, 162.159.128.12, 162.159.128.2...",13335,162.159.128.0/24,162.159.128.0/19,75,"[{'city': 'Dar es Salaam', 'country_code': 'TZ..."


In [19]:
census_critical

,prefix_x,weight,domains,ips,ases,prefix_y,backing_prefix,GCD_ICMPv4,locations
0,151.101.204.0/22,9.963849e-03,"0.testing.pinterest.com, 01.emailinboundproces...","151.101.204.102, 151.101.204.106, 151.101.204....",54113,151.101.204.0/24,151.101.204.0/22,2,"[{'city': 'New York', 'country_code': 'US', 'i..."
1,151.101.204.0/22,9.963849e-03,"0.testing.pinterest.com, 01.emailinboundproces...","151.101.204.102, 151.101.204.106, 151.101.204....",54113,151.101.205.0/24,151.101.204.0/22,2,"[{'city': 'New York', 'country_code': 'US', 'i..."
2,151.101.204.0/22,9.963849e-03,"0.testing.pinterest.com, 01.emailinboundproces...","151.101.204.102, 151.101.204.106, 151.101.204....",54113,151.101.206.0/24,151.101.204.0/22,2,"[{'city': 'New York', 'country_code': 'US', 'i..."
3,151.101.204.0/22,9.963849e-03,"0.testing.pinterest.com, 01.emailinboundproces...","151.101.204.102, 151.101.204.106, 151.101.204....",54113,151.101.207.0/24,151.101.204.0/22,2,"[{'city': 'New York', 'country_code': 'US', 'i..."
4,162.159.128.0/19,8.799750e-03,"01net.it, 02db8c58be312b9a93c947918f83eae8e006...","162.159.128.102, 162.159.128.12, 162.159.128.2...",13335,162.159.128.0/24,162.159.128.0/19,75,"[{'city': 'Dar es Salaam', 'country_code': 'TZ..."
...,...,...,...,...,...,...,...,...,...
10730,103.80.5.0/24,1.372356e-09,threatdefense.bloxone.infoblox.com,103.80.5.100,395363,103.80.5.0/24,103.80.5.0/24,8,"[{'city': 'Baltimore', 'country_code': 'US', '..."
10731,52.119.40.0/24,1.372356e-09,threatdefense.bloxone.infoblox.com,52.119.40.100,395363,52.119.40.0/24,52.119.40.0/24,8,"[{'city': 'London', 'country_code': 'GB', 'id'..."
10732,216.194.103.0/24,1.289038e-09,metaverifiedsupport.net,216.194.103.228,13150,216.194.103.0/24,216.194.103.0/24,2,"[{'city': 'Baltimore', 'country_code': 'US', '..."
10733,3.163.238.0/24,1.165329e-09,jetpacglobal.com,3.163.238.61,16509,3.163.238.0/24,3.163.238.0/24,10,"[{'city': 'Bangkok', 'country_code': 'TH', 'id..."


In [22]:
census_critical['ases'].nunique()

521

In [20]:
print(f"Number of critical GCD-confirmed /24s {len(census_critical):,}")
print(f"number of critical backing anycast prefixes {census_critical['backing_prefix'].nunique():,}")

Number of critical GCD-confirmed /24s 10,735
number of critical backing anycast prefixes 2,472


In [21]:
ipv4 = top_prefix[top_prefix['prefix'].str.contains(r'\.')]
ipv6 = top_prefix[top_prefix['prefix'].str.contains(r':')]

print(f"number of critical backing prefixes {top_prefix['prefix'].nunique():,}")
print(f"number of critical backing IPv4 prefixes {ipv4['prefix'].nunique():,}")
print(f"number of critical backing IPv6 prefixes {ipv6['prefix'].nunique():,}")


number of critical backing prefixes 134,659
number of critical backing IPv4 prefixes 121,320
number of critical backing IPv6 prefixes 13,339
